# Agentic AI Assignment Beta — Question 4

**Student ID:** 23k-6005  
**Topic:** LLM evaluation metrics

This notebook reproduces all eight calculations in Question 4. It uses only the Python standard library and the numerical inputs supplied in the assignment. Run cells from top to bottom.


## Inputs and conventions

Rows in both confusion matrices are the actual class; columns are the predicted class. The class order is **Positive, Negative, Neutral**. Confidence intervals use the 95% Wilson score interval (`z = 1.96`). For text metrics, tokenization is lowercase whitespace tokenization. METEOR uses exact token matches only (no stemming or synonym matching).


In [1]:
from collections import Counter
from math import exp, log, sqrt
from statistics import mean, stdev

labels = ["Positive", "Negative", "Neutral"]

# Each row is an actual class and each column is a predicted class.
stage1 = [
    [100, 5, 15],
    [8, 80, 12],
    [20, 10, 50],
]
stage2 = [
    [106, 4, 10],
    [6, 84, 10],
    [9, 7, 64],
]
prompt_order_accuracies = [0.847, 0.840, 0.853, 0.837, 0.850]
reference = "the delivery was late but the product quality is good"
candidate = "the product quality is good but delivery was slow"
token_probabilities = [0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20]

def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0

def wilson_interval(correct, total, z=1.96):
    p_hat = correct / total
    denominator = 1 + z ** 2 / total
    center = (p_hat + z ** 2 / (2 * total)) / denominator
    half_width = z * sqrt(p_hat * (1 - p_hat) / total + z ** 2 / (4 * total ** 2)) / denominator
    return center - half_width, center + half_width

def evaluate_matrix(matrix):
    total = sum(map(sum, matrix))
    correct = sum(matrix[i][i] for i in range(len(matrix)))
    per_class = []
    for i, label in enumerate(labels):
        tp = matrix[i][i]
        fp = sum(matrix[row][i] for row in range(len(matrix))) - tp
        fn = sum(matrix[i]) - tp
        precision = safe_divide(tp, tp + fp)
        recall = safe_divide(tp, tp + fn)
        f1 = safe_divide(2 * precision * recall, precision + recall)
        support = sum(matrix[i])
        per_class.append({"label": label, "tp": tp, "fp": fp, "fn": fn,
                          "precision": precision, "recall": recall, "f1": f1,
                          "support": support})
    macro_f1 = mean(item["f1"] for item in per_class)
    weighted_f1 = sum(item["f1"] * item["support"] for item in per_class) / total
    # For one-label multiclass classification, micro F1 equals accuracy.
    micro_f1 = correct / total
    return {"total": total, "correct": correct, "accuracy": correct / total,
            "ci": wilson_interval(correct, total), "per_class": per_class,
            "macro_f1": macro_f1, "weighted_f1": weighted_f1, "micro_f1": micro_f1}

def print_class_metrics(result):
    print(f"{'Class':<10} {'TP':>3} {'FP':>3} {'FN':>3} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    for item in result["per_class"]:
        print(f"{item['label']:<10} {item['tp']:>3} {item['fp']:>3} {item['fn']:>3} "
              f"{item['precision']:>10.5f} {item['recall']:>10.5f} {item['f1']:>10.5f}")


## Part 1 — Stage 1 accuracy and 95% confidence interval

\(	ext{Accuracy}=rac{	ext{correct predictions}}{	ext{total predictions}}\). The Wilson interval is used because it is suitable for a proportion such as accuracy.


In [2]:
result1 = evaluate_matrix(stage1)
lower1, upper1 = result1["ci"]
print(f"Correct predictions = {result1['correct']} out of {result1['total']}")
print(f"Stage 1 accuracy = {result1['accuracy']:.5f} ({result1['accuracy']:.2%})")
print(f"95% Wilson confidence interval = [{lower1:.5f}, {upper1:.5f}]")
assert result1["correct"] == 230 and result1["total"] == 300
assert round(result1["accuracy"], 4) == 0.7667


Correct predictions = 230 out of 300
Stage 1 accuracy = 0.76667 (76.67%)
95% Wilson confidence interval = [0.71562, 0.81097]


## Part 2 — Stage 1 precision, recall, and F1

For each class: \(P=rac{TP}{TP+FP}\), \(R=rac{TP}{TP+FN}\), and \(F1=rac{2PR}{P+R}\). Each class is treated as the positive class once.


In [3]:
print_class_metrics(result1)
assert round(result1["per_class"][0]["f1"], 5) == 0.80645
assert round(result1["per_class"][1]["f1"], 5) == 0.82051
assert round(result1["per_class"][2]["f1"], 5) == 0.63694


Class       TP  FP  FN  Precision     Recall         F1
Positive   100  28  20    0.78125    0.83333    0.80645
Negative    80  15  20    0.84211    0.80000    0.82051
Neutral     50  27  30    0.64935    0.62500    0.63694


## Part 3 — Stage 1 macro, weighted, and micro F1

Macro F1 gives every class equal weight. Weighted F1 weights each class by its number of actual reviews. In single-label multiclass classification, micro F1 is equal to accuracy.


In [4]:
print(f"Macro F1    = {result1['macro_f1']:.5f}")
print(f"Weighted F1 = {result1['weighted_f1']:.5f}")
print(f"Micro F1    = {result1['micro_f1']:.5f}")
assert round(result1["macro_f1"], 5) == 0.75464
assert round(result1["weighted_f1"], 5) == 0.76594
assert round(result1["micro_f1"], 5) == 0.76667


Macro F1    = 0.75464
Weighted F1 = 0.76594
Micro F1    = 0.76667


## Part 4 — Stage 1 interpretation

The model is strongest on Positive and Negative reviews. Neutral is the weakest class: its recall is 0.625, so 30 of 80 actual Neutral reviews are predicted as another class. A clearer Neutral definition and examples of mixed reviews may improve that class.


In [5]:
neutral1 = result1["per_class"][2]
missed_neutral = neutral1["fn"]
print(f"Neutral recall = {neutral1['recall']:.5f}")
print(f"Missed actual Neutral reviews = {missed_neutral} out of {neutral1['support']}")
assert missed_neutral == 30


Neutral recall = 0.62500
Missed actual Neutral reviews = 30 out of 80


## Part 5 — Stage 2 evaluation and comparison

The same calculations are repeated for the improved prompt. The final table compares the stages and highlights the change in Neutral recall.


In [6]:
result2 = evaluate_matrix(stage2)
lower2, upper2 = result2["ci"]
print(f"Correct predictions = {result2['correct']} out of {result2['total']}")
print(f"Stage 2 accuracy = {result2['accuracy']:.5f} ({result2['accuracy']:.2%})")
print(f"95% Wilson confidence interval = [{lower2:.5f}, {upper2:.5f}]\n")
print_class_metrics(result2)
print("\nMetric              Stage 1    Stage 2")
print(f"Accuracy            {result1['accuracy']:.5f}    {result2['accuracy']:.5f}")
print(f"Macro F1            {result1['macro_f1']:.5f}    {result2['macro_f1']:.5f}")
print(f"Weighted F1         {result1['weighted_f1']:.5f}    {result2['weighted_f1']:.5f}")
print(f"Micro F1            {result1['micro_f1']:.5f}    {result2['micro_f1']:.5f}")
print(f"Neutral recall      {result1['per_class'][2]['recall']:.5f}    {result2['per_class'][2]['recall']:.5f}")
print(f"\nAccuracy change = {(result2['accuracy'] - result1['accuracy']):.5f} "
      f"({(result2['accuracy'] - result1['accuracy']) * 100:.2f} percentage points)")
assert result2["correct"] == 254
assert round(result2["accuracy"], 4) == 0.8467
assert round(result2["macro_f1"], 5) == 0.84056


Correct predictions = 254 out of 300
Stage 2 accuracy = 0.84667 (84.67%)
95% Wilson confidence interval = [0.80153, 0.88303]

Class       TP  FP  FN  Precision     Recall         F1
Positive   106  15  14    0.87603    0.88333    0.87967
Negative    84  11  16    0.88421    0.84000    0.86154
Neutral     64  20  16    0.76190    0.80000    0.78049

Metric              Stage 1    Stage 2
Accuracy            0.76667    0.84667
Macro F1            0.75464    0.84056
Weighted F1         0.76594    0.84718
Micro F1            0.76667    0.84667
Neutral recall      0.62500    0.80000

Accuracy change = 0.08000 (8.00 percentage points)


## Part 6 — Prompt-order stability

Five accuracy values are given for different example orders. The sample standard deviation uses \(n-1\) in the denominator.


In [7]:
order_mean = mean(prompt_order_accuracies)
order_sd = stdev(prompt_order_accuracies)  # sample standard deviation
print("Accuracy values:", prompt_order_accuracies)
print(f"Mean = {order_mean:.5f}")
print(f"Sample standard deviation = {order_sd:.5f}")
print(f"Reported stability = {order_mean:.5f} +/- {order_sd:.5f}")
assert round(order_mean, 4) == 0.8454
assert round(order_sd, 5) == 0.00673


Accuracy values: [0.847, 0.84, 0.853, 0.837, 0.85]
Mean = 0.84540
Sample standard deviation = 0.00673
Reported stability = 0.84540 +/- 0.00673


## Part 7 — BLEU-4, ROUGE, and METEOR

BLEU uses clipped n-gram precision and a brevity penalty. ROUGE-1 and ROUGE-2 are F1 scores based on unigram and bigram overlap. ROUGE-L is based on the longest common subsequence. METEOR here follows the assignment's exact-match convention: 8 matches and 3 chunks.


In [8]:
ref_tokens = reference.split()
cand_tokens = candidate.split()

def ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def clipped_matches(reference_tokens, candidate_tokens, n):
    reference_counts = Counter(ngrams(reference_tokens, n))
    candidate_counts = Counter(ngrams(candidate_tokens, n))
    return sum((reference_counts & candidate_counts).values()), sum(candidate_counts.values())

def f1_score(precision, recall):
    return safe_divide(2 * precision * recall, precision + recall)

def lcs_length(a, b):
    previous = [0] * (len(b) + 1)
    for token_a in a:
        current = [0]
        for j, token_b in enumerate(b, start=1):
            current.append(previous[j - 1] + 1 if token_a == token_b else max(previous[j], current[-1]))
        previous = current
    return previous[-1]

precisions = []
for n in range(1, 5):
    matches, total = clipped_matches(ref_tokens, cand_tokens, n)
    precisions.append(matches / total)
    print(f"Modified {n}-gram precision = {matches}/{total} = {matches / total:.5f}")

brevity_penalty = exp(1 - len(ref_tokens) / len(cand_tokens))
bleu4 = brevity_penalty * exp(sum(log(p) for p in precisions) / 4)

unigram_matches, _ = clipped_matches(ref_tokens, cand_tokens, 1)
bigram_matches, _ = clipped_matches(ref_tokens, cand_tokens, 2)
rouge1_p = unigram_matches / len(cand_tokens)
rouge1_r = unigram_matches / len(ref_tokens)
rouge2_p = bigram_matches / (len(cand_tokens) - 1)
rouge2_r = bigram_matches / (len(ref_tokens) - 1)
rouge1 = f1_score(rouge1_p, rouge1_r)
rouge2 = f1_score(rouge2_p, rouge2_r)
lcs = lcs_length(ref_tokens, cand_tokens)
rouge_l = f1_score(lcs / len(cand_tokens), lcs / len(ref_tokens))

meteor_p, meteor_r = rouge1_p, rouge1_r
meteor_f_mean = 10 * meteor_p * meteor_r / (meteor_r + 9 * meteor_p)
meteor_chunks = 3
meteor_penalty = 0.5 * (meteor_chunks / unigram_matches) ** 3
meteor = meteor_f_mean * (1 - meteor_penalty)

print(f"\nBrevity penalty = {brevity_penalty:.5f}")
print(f"BLEU-4 = {bleu4:.5f}")
print(f"ROUGE-1 F1 = {rouge1:.5f}")
print(f"ROUGE-2 F1 = {rouge2:.5f}")
print(f"ROUGE-L F1 (LCS = {lcs}) = {rouge_l:.5f}")
print(f"METEOR exact-match F-mean = {meteor_f_mean:.5f}")
print(f"METEOR fragmentation penalty = {meteor_penalty:.5f}")
print(f"METEOR = {meteor:.5f}")
assert [round(p, 5) for p in precisions] == [0.88889, 0.625, 0.42857, 0.33333]
assert round(bleu4, 5) == 0.47495
assert round(rouge1, 5) == 0.84211 and round(rouge2, 5) == 0.58824
assert round(rouge_l, 5) == 0.52632 and round(meteor, 5) == 0.78677


Modified 1-gram precision = 8/9 = 0.88889
Modified 2-gram precision = 5/8 = 0.62500
Modified 3-gram precision = 3/7 = 0.42857
Modified 4-gram precision = 2/6 = 0.33333

Brevity penalty = 0.89484
BLEU-4 = 0.47495
ROUGE-1 F1 = 0.84211
ROUGE-2 F1 = 0.58824
ROUGE-L F1 (LCS = 5) = 0.52632
METEOR exact-match F-mean = 0.80808
METEOR fragmentation penalty = 0.02637
METEOR = 0.78677


## Part 8 — Perplexity

With token probabilities \(p_i\), perplexity is \(\exp(-rac{1}{N}\sum_i\ln p_i)\). Lower perplexity means the model was more confident in these token choices, although confidence by itself does not prove an explanation is correct.


In [9]:
log_probability_sum = sum(log(p) for p in token_probabilities)
perplexity = exp(-log_probability_sum / len(token_probabilities))
print("Token probabilities:", token_probabilities)
print(f"Number of tokens = {len(token_probabilities)}")
print(f"Sum of log probabilities = {log_probability_sum:.5f}")
print(f"Perplexity = {perplexity:.5f}")
assert len(token_probabilities) == 9
assert round(log_probability_sum, 5) == -7.59293
assert round(perplexity, 5) == 2.32486


Token probabilities: [0.6, 0.4, 0.5, 0.8, 0.7, 0.3, 0.25, 0.5, 0.2]
Number of tokens = 9
Sum of log probabilities = -7.59293
Perplexity = 2.32486


## Final interpretation

The improved Stage 2 prompt raises accuracy from 76.67% to 84.67% and improves Neutral recall from 62.50% to 80.00%. Across the five prompt-order runs, the standard deviation is small (about 0.00673), suggesting relatively stable performance. The text-metric scores compare a candidate explanation with a reference, while perplexity summarizes the confidence of the supplied token probabilities.
